# Clinical Data Preprocessing — METABRIC
Input: `data/data_clinical_patient.txt` + `data/data_clinical_sample.txt` (cBioPortal format)

Output: `clinical/statistical_filtered/clin_8_composite_Nfeatures.csv`

In [2]:
from pathlib import Path

_cwd = Path().resolve()
BASE_DIR = next(
    p for p in [_cwd, *_cwd.parents]
    if (p / "data" / "data_clinical_patient.txt").exists()
)

PATIENT_FILE = BASE_DIR / "data" / "data_clinical_patient.txt"
SAMPLE_FILE  = BASE_DIR / "data" / "data_clinical_sample.txt"
OUTCOME_FILE = BASE_DIR / "rna" / "statistical_filtered" / "outcome.csv"
OUTPUT_DIR   = BASE_DIR / "clinical" / "statistical_filtered"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert PATIENT_FILE.exists(), f'Not found: {PATIENT_FILE}'
assert SAMPLE_FILE.exists(),  f'Not found: {SAMPLE_FILE}'
assert OUTCOME_FILE.exists(), f'Not found: {OUTCOME_FILE}'

print(f'Base   : {BASE_DIR}')
print(f'Output : {OUTPUT_DIR}')
print('Paths OK')

Base   : C:\Users\olegk\Desktop\Thesis_v3\03_METABRIC_external_validation
Output : C:\Users\olegk\Desktop\Thesis_v3\03_METABRIC_external_validation\clinical\statistical_filtered
Paths OK


In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')
print('Imports OK')

Imports OK


## 1. Load cBioPortal clinical files
Lines starting with `#` are metadata — skipped automatically by `comment='#'`.

In [5]:
def read_cbio(path):
    df = pd.read_csv(path, sep='\t', comment='#', index_col=0, low_memory=False)
    print(f'  {Path(path).name}: {df.shape}')
    return df

print('Loading clinical files...')
patient = read_cbio(PATIENT_FILE)
sample  = read_cbio(SAMPLE_FILE)

print(f'\nPatient columns: {patient.columns.tolist()}')
print(f'\nSample columns : {sample.columns.tolist()}')

Loading clinical files...
  data_clinical_patient.txt: (2509, 23)
  data_clinical_sample.txt: (2509, 12)

Patient columns: ['LYMPH_NODES_EXAMINED_POSITIVE', 'NPI', 'CELLULARITY', 'CHEMOTHERAPY', 'COHORT', 'ER_IHC', 'HER2_SNP6', 'HORMONE_THERAPY', 'INFERRED_MENOPAUSAL_STATE', 'SEX', 'INTCLUST', 'AGE_AT_DIAGNOSIS', 'OS_MONTHS', 'OS_STATUS', 'CLAUDIN_SUBTYPE', 'THREEGENE', 'VITAL_STATUS', 'LATERALITY', 'RADIO_THERAPY', 'HISTOLOGICAL_SUBTYPE', 'BREAST_SURGERY', 'RFS_MONTHS', 'RFS_STATUS']

Sample columns : ['SAMPLE_ID', 'CANCER_TYPE', 'CANCER_TYPE_DETAILED', 'ER_STATUS', 'HER2_STATUS', 'GRADE', 'ONCOTREE_CODE', 'PR_STATUS', 'SAMPLE_TYPE', 'TUMOR_SIZE', 'TUMOR_STAGE', 'TMB_NONSYNONYMOUS']


## 2. Merge patient + sample tables

In [7]:
print(f'Patient index: {patient.index.name}  sample: {patient.index[:3].tolist()}')
print(f'Sample  index: {sample.index.name}   sample: {sample.index[:3].tolist()}')

if 'PATIENT_ID' in sample.columns:
    clin = sample.merge(patient, left_on='PATIENT_ID', right_index=True,
                        how='left', suffixes=('', '_patient'))
    clin = clin.drop(columns=[c for c in clin.columns if c.endswith('_patient')])
    print(f'\nMerged on PATIENT_ID: {clin.shape}')
else:
    clin = sample.copy()
    print(f'\nUsing sample file only: {clin.shape}')

print(f'Index examples: {clin.index[:5].tolist()}')

Patient index: PATIENT_ID  sample: ['MB-0000', 'MB-0002', 'MB-0005']
Sample  index: PATIENT_ID   sample: ['MB-0000', 'MB-0002', 'MB-0005']

Using sample file only: (2509, 12)
Index examples: ['MB-0000', 'MB-0002', 'MB-0005', 'MB-0006', 'MB-0008']


## 3. Align with outcome

In [9]:
outcome = pd.read_csv(OUTCOME_FILE, index_col=0)
print(f'Outcome  samples: {len(outcome)}  eg: {outcome.index[:3].tolist()}')
print(f'Clinical samples: {len(clin)}    eg: {clin.index[:3].tolist()}')

common = sorted(set(clin.index) & set(outcome.index))
print(f'Direct overlap  : {len(common)}')

if len(common) == 0:
    print('No direct overlap — check index format above and adjust manually')

clin    = clin.loc[common].copy()
outcome = outcome.loc[common].copy()
print(f'Final aligned   : {len(common)} samples')

Outcome  samples: 1980  eg: ['MB-0000', 'MB-0002', 'MB-0005']
Clinical samples: 2509    eg: ['MB-0000', 'MB-0002', 'MB-0005']
Direct overlap  : 1980
Final aligned   : 1980 samples


## 4. Quality control

In [11]:
# Drop outcome-related columns
outcome_kw = ['OS_MONTHS', 'OS_STATUS', 'VITAL', 'SURVIVAL', 'OS.TIME',
              'OVERALL_SURVIVAL', 'DISEASE_FREE', 'DFS', 'DEATH']
drop_cols = [c for c in clin.columns if any(kw in c.upper() for kw in outcome_kw)]
if drop_cols:
    print(f'Dropping outcome columns: {drop_cols}')
    clin = clin.drop(columns=drop_cols)
else:
    print('No outcome columns — OK')

# Drop ID/metadata columns
drop_id = [c for c in clin.columns if c.upper() in ['PATIENT_ID', 'SAMPLE_ID', 'STUDY_ID']]
if drop_id:
    print(f'Dropping ID columns: {drop_id}')
    clin = clin.drop(columns=drop_id)

# Missing values
miss = clin.isna().mean(axis=0).sort_values(ascending=False)
print(f'\nMissing per feature:')
print(miss[miss > 0].to_string() if (miss > 0).any() else '  None')

drop_miss = miss[miss > 0.50].index.tolist()
if drop_miss:
    print(f'\nDropping >50% missing: {drop_miss}')
    clin = clin.drop(columns=drop_miss)

print(f'\nShape after QC : {clin.shape}')
print(f'Features       : {clin.columns.tolist()}')

No outcome columns — OK
Dropping ID columns: ['SAMPLE_ID']

Missing per feature:
TUMOR_STAGE    0.259596
GRADE          0.044444
TUMOR_SIZE     0.013131

Shape after QC : (1980, 11)
Features       : ['CANCER_TYPE', 'CANCER_TYPE_DETAILED', 'ER_STATUS', 'HER2_STATUS', 'GRADE', 'ONCOTREE_CODE', 'PR_STATUS', 'SAMPLE_TYPE', 'TUMOR_SIZE', 'TUMOR_STAGE', 'TMB_NONSYNONYMOUS']


## 5. Encode categorical & impute missing

In [13]:
clin_enc = clin.copy()

cat_cols = clin_enc.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'Categorical ({len(cat_cols)}): {cat_cols}')

le = LabelEncoder()
for col in cat_cols:
    clin_enc[col] = clin_enc[col].astype(str)
    clin_enc[col] = le.fit_transform(clin_enc[col])

for col in clin_enc.columns:
    if clin_enc[col].isna().any():
        clin_enc[col] = clin_enc[col].fillna(clin_enc[col].median())

zero_var = clin_enc.columns[clin_enc.var(axis=0) == 0].tolist()
if zero_var:
    print(f'Dropping zero-variance: {zero_var}')
    clin_enc = clin_enc.drop(columns=zero_var)

print(f'\nFinal shape : {clin_enc.shape}')
print(f'NaN total   : {clin_enc.isna().sum().sum()}')
print(f'Features    : {clin_enc.columns.tolist()}')

Categorical (7): ['CANCER_TYPE', 'CANCER_TYPE_DETAILED', 'ER_STATUS', 'HER2_STATUS', 'ONCOTREE_CODE', 'PR_STATUS', 'SAMPLE_TYPE']
Dropping zero-variance: ['CANCER_TYPE', 'SAMPLE_TYPE']

Final shape : (1980, 9)
NaN total   : 0
Features    : ['CANCER_TYPE_DETAILED', 'ER_STATUS', 'HER2_STATUS', 'GRADE', 'ONCOTREE_CODE', 'PR_STATUS', 'TUMOR_SIZE', 'TUMOR_STAGE', 'TMB_NONSYNONYMOUS']


## 6. Save & verify

In [15]:
fname = f'clin_8_composite_{len(clin_enc.columns)}features.csv'
clin_enc.to_csv(OUTPUT_DIR / fname)
outcome.to_csv(OUTPUT_DIR / 'outcome.csv')

df_check = pd.read_csv(OUTPUT_DIR / fname, index_col=0)
assert list(df_check.index) == list(outcome.index), 'Index mismatch!'
assert df_check.isna().sum().sum() == 0, 'NaN found!'
assert np.isinf(df_check.select_dtypes(include=np.number).values).sum() == 0, 'Inf found!'

print(f'Saved  : {fname}')
print(f'Shape  : {df_check.shape}')
print(f'No NaN : OK')
print(f'No Inf : OK')
print(f'Index  : OK')
print(f'\nNext step: run 08_merge_modalities_METABRIC.py')

Saved  : clin_8_composite_9features.csv
Shape  : (1980, 9)
No NaN : OK
No Inf : OK
Index  : OK

Next step: run 08_merge_modalities_METABRIC.py
